# Pasketti Phonetic Track - Wav2Vec2 CTC Fine-tuning

DrivenData "On Top of Pasketti" Children's Speech Recognition Challenge

**Runtime**: GPU (T4) required. Edit > Notebook settings > T4 GPU

In [ ]:
# 1. Setup
import os
from google.colab import userdata

os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

!git clone https://github.com/yasumorishima/drivendata-comp.git
%cd drivendata-comp

In [ ]:
# 2. Install dependencies
!pip install -q -r pasketti-phonetic/requirements-train.txt
!pip install -q requests

In [ ]:
# 3. Download data from GitHub Artifact
!python scripts/colab_data_download.py \
    --artifact drivendata-phonetic-data \
    --output data/phonetic

In [ ]:
# 4. Check data
!ls -lh data/phonetic/ | head -20
!wc -l data/phonetic/train_phon_transcripts.jsonl

In [ ]:
# 5. Train
# memo: experiment description (shown in W&B)
MEMO = "v1: wav2vec2-base baseline"

!python pasketti-phonetic/train.py \
    --data_dir data/phonetic \
    --output_dir model_phonetic \
    --model_name facebook/wav2vec2-base \
    --epochs 20 \
    --batch_size 16 \
    --gradient_accumulation 4 \
    --lr 5e-5 \
    --memo "$MEMO"

In [ ]:
# 6. Check model output
!ls -lh model_phonetic/final_model/
!du -sh model_phonetic/final_model/

In [ ]:
# 7. Upload model to GitHub Release
RELEASE_TAG = "phonetic-model-v1"
RELEASE_NOTES = "wav2vec2-base CTC fine-tune"

!tar czf model_phonetic.tar.gz -C model_phonetic/final_model .
!ls -lh model_phonetic.tar.gz

!echo $GH_TOKEN | gh auth login --with-token
!gh release create {RELEASE_TAG} model_phonetic.tar.gz \
    --repo yasumorishima/drivendata-comp \
    --title "{RELEASE_TAG}" \
    --notes "{RELEASE_NOTES}"

## Next: Package Submission

```bash
gh workflow run "Package DrivenData Submission" \
  --repo yasumorishima/drivendata-comp \
  -f competition_dir=pasketti-phonetic \
  -f model_release_tag=phonetic-model-v1 \
  -f memo="v1: baseline submission"
```